# 03 - Construccion OBT (One Big Table)

Objetivos:
- Consolidar un dataset analitico unico para BI y modelado.
- Agregar features temporales, geograficas y de negocio.
- Publicar la OBT en Snowflake para consumo en notebooks 04 y 05.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os

In [ ]:
app = (
    SparkSession.builder
    .appName('03_construccion_obt')
    .config('spark.sql.session.timeZone', 'UTC')
    .getOrCreate()
)

# Credenciales Snowflake desde variables de ambiente
sf_option = {
    'sfURL': os.environ['SF_HOST'],
    'sfUser': os.environ['SF_USER'],
    'sfPassword': os.environ['SF_PASSWORD'],
    'sfDatabase': os.environ['SF_DATABASE'],
    'sfSchema': os.environ.get('SF_RAW_SCHEMA', 'raw'),
    'sfWarehouse': os.environ.get('SF_WAREHOUSE', ''),
    'sfRole': os.environ.get('SF_ROLE', ''),
}

In [ ]:
# Base curada del notebook 02
df_enriched = (
    app.read.format('snowflake')
    .options(**sf_option)
    .option('dbtable', 'CURATED.FCT_TRIPS_ENRICHED')
    .load()
)

print('Rows CURATED.FCT_TRIPS_ENRICHED:', df_enriched.count())

In [ ]:
# Reglas de calidad minimas para la OBT
df_base = (
    df_enriched
    .filter(F.col('trip_nk').isNotNull())
    .filter(F.col('pickup_datetime_utc').isNotNull())
    .filter(F.col('dropoff_datetime_utc').isNotNull())
    .filter(F.col('total_amount').isNotNull())
    .filter(F.col('trip_duration_minutes').isNotNull())
    .filter(F.col('trip_duration_minutes') > 0)
)

print('Rows after quality filters:', df_base.count())

In [ ]:
# Features de tiempo, monetarias y de ruta
df_obt = (
    df_base
    .withColumn('trip_year', F.year('pickup_datetime_utc'))
    .withColumn('trip_month', F.month('pickup_datetime_utc'))
    .withColumn('trip_day', F.dayofmonth('pickup_datetime_utc'))
    .withColumn('trip_day_of_week', F.date_format('pickup_datetime_utc', 'E'))
    .withColumn('pickup_hour', F.hour('pickup_datetime_utc'))
    .withColumn(
        'pickup_time_band',
        F.when((F.col('pickup_hour') >= 6) & (F.col('pickup_hour') < 12), F.lit('morning'))
        .when((F.col('pickup_hour') >= 12) & (F.col('pickup_hour') < 18), F.lit('afternoon'))
        .when((F.col('pickup_hour') >= 18) & (F.col('pickup_hour') < 24), F.lit('evening'))
        .otherwise(F.lit('night'))
    )
    .withColumn('is_weekend', F.when(F.dayofweek('pickup_datetime_utc').isin([1, 7]), F.lit(1)).otherwise(F.lit(0)))
    .withColumn('trip_distance_miles', F.col('trip_distance').cast('double'))
    .withColumn('trip_distance_km', F.col('trip_distance').cast('double') * F.lit(1.60934))
    .withColumn(
        'distance_bucket',
        F.when(F.col('trip_distance_miles') < 1, F.lit('<1 mi'))
        .when((F.col('trip_distance_miles') >= 1) & (F.col('trip_distance_miles') < 3), F.lit('1-3 mi'))
        .when((F.col('trip_distance_miles') >= 3) & (F.col('trip_distance_miles') < 7), F.lit('3-7 mi'))
        .when((F.col('trip_distance_miles') >= 7) & (F.col('trip_distance_miles') < 15), F.lit('7-15 mi'))
        .otherwise(F.lit('15+ mi'))
    )
    .withColumn('avg_speed_mph', F.col('trip_distance_miles') / (F.col('trip_duration_minutes') / F.lit(60.0)))
    .withColumn('fare_per_mile', F.col('fare_amount') / F.when(F.col('trip_distance_miles') > 0, F.col('trip_distance_miles')).otherwise(F.lit(None)))
    .withColumn('tip_pct', (F.col('tip_amount') / F.when(F.col('fare_amount') > 0, F.col('fare_amount')).otherwise(F.lit(None))) * F.lit(100.0))
    .withColumn('route_key', F.concat_ws('->', F.coalesce(F.col('pu_zone'), F.lit('UNKNOWN')), F.coalesce(F.col('do_zone'), F.lit('UNKNOWN'))))
    .withColumn(
        'obt_trip_sk',
        F.sha2(
            F.concat_ws('||', F.col('trip_nk').cast('string'), F.col('trip_date').cast('string'), F.col('taxi_type').cast('string')),
            256
        )
    )
)

In [ ]:
# Orden final de columnas para consumo analitico
obt_columns = [
    'obt_trip_sk', 'trip_nk',
    'taxi_type', 'service_type',
    'trip_date', 'trip_year', 'trip_month', 'trip_day', 'trip_day_of_week',
    'pickup_datetime_utc', 'dropoff_datetime_utc', 'pickup_hour', 'pickup_time_band', 'is_weekend',
    'PULocationID', 'pu_borough', 'pu_zone', 'pu_service_zone',
    'DOLocationID', 'do_borough', 'do_zone', 'do_service_zone',
    'route_key',
    'vendor_id', 'vendor_desc',
    'rate_code_id', 'rate_code_desc',
    'payment_type_code', 'payment_type_desc',
    'trip_type_code', 'trip_type_desc',
    'store_and_fwd_flag_norm', 'store_and_fwd_desc',
    'passenger_count',
    'trip_distance_miles', 'trip_distance_km', 'distance_bucket', 'trip_duration_minutes', 'avg_speed_mph',
    'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'congestion_surcharge', 'airport_fee', 'total_amount',
    'fare_per_mile', 'tip_pct',
    'source_year', 'source_month', 'run_id', 'ingested_at_utc'
]

for c in obt_columns:
    if c not in df_obt.columns:
        df_obt = df_obt.withColumn(c, F.lit(None))

df_obt_final = df_obt.select(*obt_columns)
print('Rows OBT final:', df_obt_final.count())

In [ ]:
# Publicacion de la OBT principal
(
    df_obt_final
    .write.format('snowflake')
    .options(**sf_option)
    .option('dbtable', 'ANALYTICS.OBT_TRIPS')
    .mode('overwrite')
    .save()
)

In [ ]:
# Agregado mensual de apoyo para dashboards
df_obt_monthly = (
    df_obt_final
    .groupBy('trip_year', 'trip_month', 'taxi_type', 'pu_borough', 'do_borough', 'payment_type_desc')
    .agg(
        F.count('*').alias('trips_count'),
        F.sum('total_amount').alias('total_revenue'),
        F.avg('total_amount').alias('avg_ticket'),
        F.avg('trip_distance_miles').alias('avg_distance_miles'),
        F.avg('trip_duration_minutes').alias('avg_duration_minutes'),
        F.avg('tip_pct').alias('avg_tip_pct')
    )
)

(
    df_obt_monthly
    .write.format('snowflake')
    .options(**sf_option)
    .option('dbtable', 'ANALYTICS.OBT_TRIPS_MONTHLY')
    .mode('overwrite')
    .save()
)

In [ ]:
# Validaciones rapidas
df_obt_final.groupBy('taxi_type').count().orderBy('taxi_type').show()

df_obt_final.select(
    F.count('*').alias('rows_obt'),
    F.countDistinct('trip_nk').alias('distinct_trip_nk'),
    F.sum(F.when(F.col('pu_zone').isNull(), 1).otherwise(0)).alias('null_pu_zone'),
    F.sum(F.when(F.col('do_zone').isNull(), 1).otherwise(0)).alias('null_do_zone')
).show()

df_obt_final.select(
    'trip_nk', 'taxi_type', 'trip_date', 'pickup_time_band',
    'pu_borough', 'pu_zone', 'do_borough', 'do_zone',
    'distance_bucket', 'trip_distance_miles', 'trip_duration_minutes', 'avg_speed_mph',
    'payment_type_desc', 'total_amount', 'tip_pct', 'route_key'
).show(20, truncate=False)